# 13 · D2b: training objectives with the same batches (supplement R5 / T1)

**Question:** in D2a (`06_d2_losses`) MSE and ce4 were trained on random batches of pairs, while ListNet was trained on batches of query lists cut to at most 16 candidates, with a slightly different warm-up share. Does the ranking of the objectives hold when **every loss sees the same batches** (whole query lists, never cut) and the same warm-up share? D2a is kept as it is; D2b runs next to it.

| Run | Loss | Sampler | Seeds | Compared with |
|---|---|---|---|---|
| `d2q_minilm_title_mse` | MSE on the gain | whole queries | 42–45 | reference of D2b |
| `d2q_minilm_title_ce4` | 4-class cross-entropy → expected gain | whole queries | 42–45 | `d2q_minilm_title_mse`, same seed |
| `d2q_minilm_title_listnet` | ListNet, target gain / sum(gain) | whole queries | 42–45 | `d2q_minilm_title_mse`, same seed |

**Settings** = B3 except the batches: `cross-encoder/ms-marco-MiniLM-L-12-v2`, `title`, `max_len` 128, 1 epoch, batch ≤ 32 pairs, lr 7e-6, fp16; `sampler='query'`, `list_size=None` (a query is never split; a query with more than 32 candidates is a batch of its own); `warmup_ratio=0.42` (B3 and ce4 used 4,962 of 11,816 steps = 42 %; D2a ListNet used 5,000 of 12,883 = 38.8 %). MSE and ce4 keep queries with one candidate; ListNet skips them (their loss is 0).

**Reading** (section 3): within D2b, a loss beats MSE when the val difference is positive for all 4 seeds and the CI of the difference pooled over seeds excludes 0. Across versions (seeds 42, 43): `d2q_*` vs D2a (`b3_*`, `d2_*_ce4`, `d2_*_listnet`) measures the effect of the batching itself. Never compare a D2b loss with a D2a loss of another kind.

**Test is blind:** test scores are saved, not scored; `10_report` scores them after the supplement list is frozen (29/11).

**Kaggle settings:** GPU (every visible GPU is used: one run per GPU at a time, `src/gpu_jobs.py`; with one GPU the runs go one after the other). **Internet on**, or attach the output of `00_hf_models`.

**Inputs:**
- Dataset `wm-src` built from the **`supplement`** branch (has `sampler`, `warmup_ratio`, `make_epochs` in `src/cross_encoder.py` and `src/gpu_jobs.py`). Required.
- Optional, for the cross-version comparison (D2b vs D2a): Notebook Output → **03_cross_encoder** (the original version: `scores/b3_minilm_title_mse_s{42,43}_val.parquet`) and **06_d2_losses** (`scores/d2_minilm_title_{ce4,listnet}_s{42,43}_val.parquet`). Without them that part is skipped.
- Add Input → Notebook Output → **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.
- Optional: Notebook Output → **00_hf_models** (local copy of the model, for a session without Internet).

**How to run.**
1. `MODE = 'smoke'`: 5 % of train, seed 42, val only, the three losses in parallel. Check section 0 and the verdict of section 2 (it prints the time of one full run).
2. `MODE = 'full'`, then **Save Version → Save & Run All** (12 runs = 3 losses × 4 seeds, in rounds of one run per GPU).
3. Download `results/` into `data/results/supplement/` of the repo (scores stay on Kaggle, see `.gitignore`).

**Output** (`/kaggle/working/results/`): `<run>.json` (val metrics, config, loss log, `sampler`, `mean_batch_pairs`, `src_version`), `scores/<run>_{val,test}.parquet` (ce4 also with `p_E..p_I`), `d2b_summary.csv`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import os

import numpy as np
import pandas as pd
import torch
import transformers

import src.cross_encoder as ce
from src.data import load_pairs
from src.evaluation import RunLog, calibration, paired_bootstrap

assert hasattr(ce, 'make_epochs') and hasattr(ce, 'warmup_of'), 'wm-src is not from the supplement branch: rebuild it'
from src.gpu_jobs import cleanup, prefetch, run_jobs, save_frame

transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': 5% of train, seed 42, val only | 'full': all train, seeds 42-45

CFG = {'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'title', 'max_len': 128,
       'epochs': 1, 'batch_size': 32, 'lr': 7e-6, 'warmup_steps': 5000, 'warmup_ratio': 0.42,
       'weight_decay': 0.01, 'max_grad_norm': 1.0, 'fp16': True,
       'sampler': 'query', 'list_size': None}                     # = B3 except the batches and the warm-up share
LOSSES = ['mse', 'ce4', 'listnet']                                 # mse first: it is the reference of the others
SEEDS = [42] if MODE == 'smoke' else [42, 43, 44, 45]
D2A = {'mse': 'b3_minilm_title_mse_s{seed}', 'ce4': 'd2_minilm_title_ce4_s{seed}',
       'listnet': 'd2_minilm_title_listnet_s{seed}'}               # D2a runs (seeds 42, 43 only)

SEED = 42
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']     # test is predicted and saved, never scored here
EVAL_EVERY = 100 if MODE == 'smoke' else None                      # full: no val curve (saves ~2 min per run)
PREFIX = 'smoke_' if MODE == 'smoke' else ''
RES = Path('/kaggle/working/results')
WORK = Path('/kaggle/working/jobs')                                 # job logs (small)
DATA = Path('/tmp/wm_jobs_data')                                  # temporary data copies (not in the output)

assert MODE in ('smoke', 'full')
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU'
N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', N_GPU, 'x', torch.cuda.get_device_name(0))
print('MODE', MODE, '| losses', LOSSES, '| seeds', SEEDS)

## 0. Batching self-check

On a small hand-made set of query ids (CPU): with `sampler='query'` and `list_size=None`, every row is used once per epoch for MSE and ce4, each query sits whole in one list, and the warm-up share is 42 %. The same checks run in `tests/test_cross_encoder.py`.

In [ ]:
qids = np.repeat(np.arange(80), np.random.default_rng(0).integers(1, 45, 80))   # includes 1-pair queries
toy = pd.DataFrame({'query_id': qids, 'esci_label': 'E', 'gain': 1.0})
for loss in LOSSES:
    cfg = {**CFG, 'loss': loss}
    batches = ce.make_epochs(toy, cfg, seed=0)[0]
    rows = np.concatenate([i for i, _ in batches])
    sizes = pd.Series(qids).value_counts()
    expected = len(toy) if loss != 'listnet' else sizes[sizes > 1].sum()
    assert len(rows) == len(set(rows)) == expected, loss
    for i, s in batches:
        for k in range(s.max() + 1):
            q = set(qids[i[s == k]])
            assert len(q) == 1 and (s == k).sum() == sizes[q.pop()], f'{loss}: a query was split'
assert ce.warmup_of(CFG, 11816) == 4962 and ce.warmup_of({'warmup_steps': 5000}, 11816) == 4962
print('batching self-check passed |', len(batches), 'batches for', len(rows), 'rows (listnet)')

## 1. Data

Same pairs as every run; product `title`. In smoke mode, 5 % of the training queries.

In [ ]:
pairs = load_pairs(CFG['fields'])
MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
train_df = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train_df = train_df[train_df['query_id'].isin(keep)]
train_df = train_df.reset_index(drop=True)
parts = {s: pairs[MASK[s]] for s in EVAL_SPLITS}
print(f'train {train_df["query_id"].nunique():,} queries / {len(train_df):,} pairs | '
      f'val {MASK["val"].sum():,} | test {MASK["test"].sum():,} pairs')

## 2. Training runs

Each run is one job of `src/gpu_jobs.py`: `train_ce` with `sampler='query'`, then the val / test prediction, on one GPU. The jobs run in parallel, one per GPU; their output is printed with the run name in front. When all are done, every run is logged here in the table order (MSE first): val metrics and the paired bootstrap against the MSE run of the same seed by `RunLog`; test scores saved blind. ce4 runs also save `p_E..p_I` (for calibration).

In [ ]:
prefetch(CFG['model'])
data = {'train': save_frame(train_df, DATA / 'train.parquet'),
        **{s: save_frame(parts[s], DATA / f'{s}.parquet') for s in EVAL_SPLITS}}
jobs = [{'name': f'{PREFIX}d2q_minilm_title_{loss}_s{seed}', 'kind': 'train', 'cfg': {**CFG, 'loss': loss},
         'seed': seed, 'train': data['train'], 'predict': {s: data[s] for s in EVAL_SPLITS},
         'eval_every': EVAL_EVERY, 'print_every': 100 if MODE == 'smoke' else 1000}
        for loss in LOSSES for seed in SEEDS]
results = run_jobs(jobs, WORK, N_GPU)

log = RunLog(RES, {s: parts[s][['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
runs, probs_val = [], {}
for job in jobs:
    name, cfg, seed, r = job['name'], job['cfg'], job['seed'], results[job['name']]
    loss = cfg['loss']
    ref = f'{PREFIX}d2q_minilm_title_mse_s{seed}' if loss != 'mse' else None
    extra = r['probs'] if loss == 'ce4' else None
    if extra:
        probs_val[name] = extra['val']
    log.log(name, r['scores'], {'mode': MODE, **cfg, 'seed': seed, 'subsample': SUBSAMPLE,
                                'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
                                'parallel_jobs': N_GPU, **r['info']},
            compare_to=ref, blind=('test',), extra=extra)
    runs.append(name)
cleanup(WORK, DATA)

In [ ]:
if MODE == 'smoke':
    ok = True
    for n in runs:
        r = log.records[n]
        first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
        v = r['metrics']['val']['ndcg']
        whole = r['train_pairs_used'] == r['train_pairs'] if '_listnet_' not in n else r['train_pairs_used'] <= r['train_pairs']
        checks = {'loss down': last < first, 'val above random (0.742)': v > 0.742,
                  'batches of 10-32 pairs on average': 10 <= r['mean_batch_pairs'] <= 32,
                  'every pair used (mse, ce4)': whole, 'warm-up 42 %': r['warmup'] == int(0.42 * r['steps'])}
        good = all(checks.values())
        ok &= good
        print(f"{'OK   ' if good else 'CHECK'} {n:36s} steps {r['steps']:>5} | loss {first:.4f} -> {last:.4f} | "
              f"val nDCG {v} | {r['mean_batch_pairs']} pairs/batch | full epoch ~{r['steps'] / SUBSAMPLE / r['it_per_s'] / 3600:.2f} h"
              + ('' if good else f' | failed: {[c for c, g in checks.items() if not g]}'))
    print('Smoke test passed: switch MODE to full.' if ok else 'Some checks failed: send me this output.')

## 3. Results (val)

1. Each run: val metrics and the paired difference with the MSE run of the same seed.
2. **D2b, pooled over seeds:** per query, the mean over seeds of (loss − MSE), bootstrap over queries; plus the difference per seed and whether all seeds agree in sign.
3. **Batching effect** (seeds 42, 43): each D2b run against the D2a run of the same loss and seed, if the D2a scores are in `wm-src`.
4. Calibration of the ce4 runs (NLL, Brier, classwise ECE; MSE and ListNet have no class probabilities: N/A).

In [ ]:
def pooled_delta(pairs_of_runs, split='val', metric='ndcg', n_boot=1000, seed=42):
    """Paired difference b - a pooled over seeds: per query, the mean over seeds of (b_s - a_s); bootstrap over
    queries. pairs_of_runs: [(run_a, run_b), ...], one pair per seed, all in log.per_query.
    Returns the pooled diff and CI, the diff per seed and whether every seed has the same sign."""
    per_seed, frames = [], []
    for a, b in pairs_of_runs:
        p = paired_bootstrap(log.per_query[a][split], log.per_query[b][split], metric)
        per_seed.append(p['diff'])
        d = log.per_query[a][split][['query_id', metric]].merge(
            log.per_query[b][split][['query_id', metric]], on='query_id', suffixes=('_a', '_b')).dropna()
        frames.append(d.set_index('query_id').eval(f'{metric}_b - {metric}_a').rename(len(frames)))
    diff = pd.concat(frames, axis=1, join='inner').mean(axis=1).to_numpy()
    means = diff[np.random.default_rng(seed).integers(0, len(diff), (n_boot, len(diff)))].mean(axis=1)
    return {'diff': round(float(diff.mean()), 4),
            'ci95': [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)],
            'per_seed': per_seed, 'same_sign': len({np.sign(x) for x in per_seed}) == 1, 'n_queries': len(diff)}

In [ ]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
log.print_paired()

rows = []
for loss in [l for l in LOSSES if l != 'mse']:
    r = pooled_delta([(f'{PREFIX}d2q_minilm_title_mse_s{s}', f'{PREFIX}d2q_minilm_title_{loss}_s{s}') for s in SEEDS])
    rows.append({'comparison': f'D2b {loss} - mse', **r})
for loss in LOSSES:
    have = [s for s in SEEDS if s in (42, 43) and log.load(D2A[loss].format(seed=s), splits=['val'])]
    if have:
        r = pooled_delta([(D2A[loss].format(seed=s), f'{PREFIX}d2q_minilm_title_{loss}_s{s}') for s in have])
        rows.append({'comparison': f'batching: D2b - D2a, {loss} (seeds {have})', **r})
    else:
        print(f'{loss}: D2a scores not in the inputs, batching effect skipped')
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(RES / f'{PREFIX}d2b_summary.csv', index=False)

cal = pd.DataFrame([{'run': n, **calibration(p.to_numpy(), parts['val']['esci_label'].to_numpy())}
                    for n, p in probs_val.items()])
display(cal)